# 🐘 Hadoop WordCount en Google Colab

Este notebook instala Hadoop en el entorno de Colab y ejecuta el ejemplo clásico de **WordCount** usando MapReduce.

**Pasos:**
1. Instalar Java y Hadoop
2. Configurar variables de entorno
3. Iniciar HDFS
4. Ejecutar WordCount

## 1. Instalar Java y Descargar Hadoop

In [1]:
%%bash
set -e
apt-get update -y > /dev/null
apt-get install -y openjdk-11-jdk ssh rsync > /dev/null
java -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


In [2]:
# Descargar e instalar Hadoop 3.3.6
import os

HADOOP_VERSION = "3.3.6"
HADOOP_URL = f"https://downloads.apache.org/hadoop/common/hadoop-{HADOOP_VERSION}/hadoop-{HADOOP_VERSION}.tar.gz"

!wget -q {HADOOP_URL} -O hadoop.tar.gz
!tar -xzf hadoop.tar.gz -C /usr/local/
!ln -sf /usr/local/hadoop-{HADOOP_VERSION} /usr/local/hadoop
print("✅ Hadoop descargado e instalado")

✅ Hadoop descargado e instalado


## 2. Configurar Variables de Entorno

In [3]:
import os

# Java
JAVA_HOME="/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["JAVA_HOME"] = JAVA_HOME

# Hadoop
HADOOP_HOME = "/usr/local/hadoop"
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["HADOOP_CONF_DIR"] = f"{HADOOP_HOME}/etc/hadoop"
os.environ["PATH"] = f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]

# Verificar
!hadoop version

Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /usr/local/hadoop-3.3.6/share/hadoop/common/hadoop-common-3.3.6.jar


## 3. Configurar Hadoop en Modo Pseudo-distribuido

In [4]:
# Escribir variables necesarias en hadoop-env.sh
hadoop_env = "/usr/local/hadoop/etc/hadoop/hadoop-env.sh"
with open(hadoop_env, "a") as f:
    f.write(f"\nexport JAVA_HOME={JAVA_HOME}\n")
    # Necesario cuando Hadoop corre como root (ej: Google Colab)
    f.write("\nexport HDFS_NAMENODE_USER=root\n")
    f.write("export HDFS_DATANODE_USER=root\n")
    f.write("export HDFS_SECONDARYNAMENODE_USER=root\n")
    f.write("export YARN_RESOURCEMANAGER_USER=root\n")
    f.write("export YARN_NODEMANAGER_USER=root\n")

print("✅ hadoop-env.sh configurado")

✅ hadoop-env.sh configurado


In [5]:
# Configurar core-site.xml
core_site = """<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
</configuration>
"""
with open("/usr/local/hadoop/etc/hadoop/core-site.xml", "w") as f:
    f.write(core_site)

print("✅ core-site.xml configurado")

✅ core-site.xml configurado


In [6]:
# Configurar hdfs-site.xml
hdfs_site = """<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>/tmp/hadoop/namenode</value>
  </property>
  <property>
    <name>dfs.datanode.data.dir</name>
    <value>/tmp/hadoop/datanode</value>
  </property>
</configuration>
"""
with open("/usr/local/hadoop/etc/hadoop/hdfs-site.xml", "w") as f:
    f.write(hdfs_site)

print("✅ hdfs-site.xml configurado")

✅ hdfs-site.xml configurado


In [7]:
# Configurar mapred-site.xml (con límites de memoria para Colab)
mapred_site = """<configuration>
  <property>
    <name>mapreduce.framework.name</name>
    <value>yarn</value>
  </property>
  <property>
    <name>mapreduce.application.classpath</name>
    <value>$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/*:$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/lib/*</value>
  </property>
  <!-- Memoria para los procesos Map y Reduce -->
  <property>
    <name>mapreduce.map.memory.mb</name>
    <value>512</value>
  </property>
  <property>
    <name>mapreduce.reduce.memory.mb</name>
    <value>512</value>
  </property>
  <!-- Heap JVM para Map y Reduce -->
  <property>
    <name>mapreduce.map.java.opts</name>
    <value>-Xmx410m</value>
  </property>
  <property>
    <name>mapreduce.reduce.java.opts</name>
    <value>-Xmx410m</value>
  </property>
</configuration>
"""
with open("/usr/local/hadoop/etc/hadoop/mapred-site.xml", "w") as f:
    f.write(mapred_site)

print("✅ mapred-site.xml configurado")

✅ mapred-site.xml configurado


In [8]:
# Configurar yarn-site.xml (con límites de memoria para Colab)
yarn_site = """<configuration>
  <property>
    <name>yarn.nodemanager.aux-services</name>
    <value>mapreduce_shuffle</value>
  </property>
  <property>
    <name>yarn.nodemanager.env-whitelist</name>
    <value>JAVA_HOME,HADOOP_COMMON_HOME,HADOOP_HDFS_HOME,HADOOP_CONF_DIR,CLASSPATH_PREPEND_DISTCACHE,HADOOP_YARN_HOME,HADOOP_HOME,PATH,LANG,TZ,HADOOP_MAPRED_HOME</value>
  </property>
  <!-- Memoria total disponible en el NodeManager -->
  <property>
    <name>yarn.nodemanager.resource.memory-mb</name>
    <value>2048</value>
  </property>
  <!-- Memoria máxima para un único contenedor -->
  <property>
    <name>yarn.scheduler.maximum-allocation-mb</name>
    <value>2048</value>
  </property>
  <!-- Memoria mínima por contenedor -->
  <property>
    <name>yarn.scheduler.minimum-allocation-mb</name>
    <value>256</value>
  </property>
  <!-- Deshabilitar comprobación de memoria virtual (falla en Colab) -->
  <property>
    <name>yarn.nodemanager.vmem-check-enabled</name>
    <value>false</value>
  </property>
  <property>
    <name>yarn.nodemanager.pmem-check-enabled</name>
    <value>false</value>
  </property>
</configuration>
"""
with open("/usr/local/hadoop/etc/hadoop/yarn-site.xml", "w") as f:
    f.write(yarn_site)

print("✅ yarn-site.xml configurado")

✅ yarn-site.xml configurado


## 4. Configurar SSH (necesario para iniciar Hadoop)

In [9]:
# Instalar y configurar SSH
!apt-get install -y ssh pdsh -qq > /dev/null

# Iniciar el servicio SSH
!service ssh start

# Generar clave SSH sin passphrase (si no existe)
import os
if not os.path.exists('/root/.ssh/id_rsa'):
    !ssh-keygen -t rsa -P '' -f ~/.ssh/id_rsa -q

!cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/authorized_keys

# Agregar localhost al known_hosts para evitar el prompt interactivo
!ssh-keyscan -H localhost >> ~/.ssh/known_hosts 2>/dev/null
!ssh-keyscan -H 0.0.0.0 >> ~/.ssh/known_hosts 2>/dev/null
!ssh-keyscan -H $(hostname) >> ~/.ssh/known_hosts 2>/dev/null
!ssh-keyscan -H $(hostname -f) >> ~/.ssh/known_hosts 2>/dev/null

# Verificar que SSH funciona antes de continuar
result = !ssh -o StrictHostKeyChecking=no -o ConnectTimeout=5 localhost echo 'SSH OK'
print('\n'.join(result))

# Forzar que Hadoop use SSH (no pdsh) para evitar problemas de conexión
import os
os.environ['PDSH_RCMD_TYPE'] = 'ssh'
!echo 'export PDSH_RCMD_TYPE=ssh' >> /usr/local/hadoop/etc/hadoop/hadoop-env.sh

print('\n✅ SSH configurado correctamente')

 * Starting OpenBSD Secure Shell server sshd
   ...done.
SSH OK

✅ SSH configurado correctamente


## 5. Formatear HDFS e Iniciar Servicios

In [10]:
# Crear directorios necesarios
!mkdir -p /tmp/hadoop/namenode /tmp/hadoop/datanode

# Formatear el NameNode (solo la primera vez)
!hdfs namenode -format -force -nonInteractive
print("✅ NameNode formateado")

2026-03-02 06:46:25,482 INFO namenode.NameNode: STARTUP_MSG: 
/************************************************************
STARTUP_MSG: Starting NameNode
STARTUP_MSG:   host = a08eacd1b88c/172.28.0.12
STARTUP_MSG:   args = [-format, -force, -nonInteractive]
STARTUP_MSG:   version = 3.3.6
STARTUP_MSG:   classpath = /usr/local/hadoop/etc/hadoop:/usr/local/hadoop/share/hadoop/common/lib/commons-net-3.9.0.jar:/usr/local/hadoop/share/hadoop/common/lib/kerb-common-1.0.1.jar:/usr/local/hadoop/share/hadoop/common/lib/netty-transport-classes-kqueue-4.1.89.Final.jar:/usr/local/hadoop/share/hadoop/common/lib/netty-transport-rxtx-4.1.89.Final.jar:/usr/local/hadoop/share/hadoop/common/lib/jetty-http-9.4.51.v20230217.jar:/usr/local/hadoop/share/hadoop/common/lib/re2j-1.1.jar:/usr/local/hadoop/share/hadoop/common/lib/avro-1.7.7.jar:/usr/local/hadoop/share/hadoop/common/lib/jcip-annotations-1.0-1.jar:/usr/local/hadoop/share/hadoop/common/lib/jaxb-api-2.2.11.jar:/usr/local/hadoop/share/hadoop/common/l

In [11]:
import os, time

# PDSH_RCMD_TYPE=ssh es clave: evita que pdsh use rsh (que falla en Colab)
os.environ['PDSH_RCMD_TYPE'] = 'ssh'

# Iniciar HDFS
!PDSH_RCMD_TYPE=ssh start-dfs.sh

time.sleep(8)

# Verificar procesos activos
print('\n🔍 Procesos Java activos (jps):')
!jps

Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [a08eacd1b88c]

🔍 Procesos Java activos (jps):
2562 NameNode
2850 SecondaryNameNode
2660 DataNode
3109 Jps


In [12]:
import time

# Iniciar YARN
!PDSH_RCMD_TYPE=ssh start-yarn.sh

time.sleep(8)

print('\n🔍 Procesos Java activos (jps):')
!jps

Starting resourcemanager
Starting nodemanagers

🔍 Procesos Java activos (jps):
3745 Jps
2562 NameNode
2850 SecondaryNameNode
2660 DataNode
3339 NodeManager
3228 ResourceManager


## 6. Preparar los Datos de Entrada

In [13]:
# Crear archivos de texto de ejemplo
texto1 = """
Hadoop is an open source framework for distributed storage and processing.
Hadoop uses MapReduce to process large datasets across clusters of computers.
Hadoop is written in Java and is used by many companies worldwide.
The Hadoop ecosystem includes tools like Hive, Pig, HBase, and Spark.
"""

texto2 = """
MapReduce is a programming model for processing large data sets.
The Map function processes input data and emits key value pairs.
The Reduce function aggregates the values for each key.
WordCount is the hello world of MapReduce and Hadoop programming.
"""

with open("/tmp/input1.txt", "w") as f:
    f.write(texto1)

with open("/tmp/input2.txt", "w") as f:
    f.write(texto2)

print("✅ Archivos de entrada creados")
print("\nContenido de input1.txt:")
print(texto1)
print("Contenido de input2.txt:")
print(texto2)

✅ Archivos de entrada creados

Contenido de input1.txt:

Hadoop is an open source framework for distributed storage and processing.
Hadoop uses MapReduce to process large datasets across clusters of computers.
Hadoop is written in Java and is used by many companies worldwide.
The Hadoop ecosystem includes tools like Hive, Pig, HBase, and Spark.

Contenido de input2.txt:

MapReduce is a programming model for processing large data sets.
The Map function processes input data and emits key value pairs.
The Reduce function aggregates the values for each key.
WordCount is the hello world of MapReduce and Hadoop programming.



In [14]:
# Crear directorio de entrada en HDFS y copiar archivos
!hdfs dfs -mkdir -p /user/root/wordcount/input
!hdfs dfs -put /tmp/input1.txt /user/root/wordcount/input/
!hdfs dfs -put /tmp/input2.txt /user/root/wordcount/input/

# Verificar archivos en HDFS
print("📁 Archivos en HDFS:")
!hdfs dfs -ls /user/root/wordcount/input/

📁 Archivos en HDFS:
Found 2 items
-rw-r--r--   1 root supergroup        291 2026-03-02 06:47 /user/root/wordcount/input/input1.txt
-rw-r--r--   1 root supergroup        253 2026-03-02 06:47 /user/root/wordcount/input/input2.txt


## 7. Ejecutar WordCount con MapReduce

In [15]:
# Ejecutar el ejemplo de WordCount incluido en Hadoop
!hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar \
    wordcount \
    /user/root/wordcount/input \
    /user/root/wordcount/output

2026-03-02 06:47:26,477 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-03-02 06:47:26,999 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1772434026935_0001
2026-03-02 06:47:27,438 INFO input.FileInputFormat: Total input files to process : 2
2026-03-02 06:47:28,371 INFO mapreduce.JobSubmitter: number of splits:2
2026-03-02 06:47:28,654 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1772434026935_0001
2026-03-02 06:47:28,655 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-03-02 06:47:28,908 INFO conf.Configuration: resource-types.xml not found
2026-03-02 06:47:28,909 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-03-02 06:47:29,268 INFO impl.YarnClientImpl: Submitted application application_1772434026935_0001
2026-03-02 06:47:29,382 INFO mapreduce.Job: The url to track the job: http://a08eacd1b88c:8088/proxy/application_1

## 8. Ver los Resultados

In [16]:
# Listar archivos de salida
print("📁 Archivos de salida en HDFS:")
!hdfs dfs -ls /user/root/wordcount/output/

print("\n📊 Resultado del WordCount:")
!hdfs dfs -cat /user/root/wordcount/output/part-r-00000

📁 Archivos de salida en HDFS:
Found 2 items
-rw-r--r--   1 root supergroup          0 2026-03-02 06:47 /user/root/wordcount/output/_SUCCESS
-rw-r--r--   1 root supergroup        547 2026-03-02 06:47 /user/root/wordcount/output/part-r-00000

📊 Resultado del WordCount:
HBase,	1
Hadoop	5
Hive,	1
Java	1
Map	1
MapReduce	3
Pig,	1
Reduce	1
Spark.	1
The	3
WordCount	1
a	1
across	1
aggregates	1
an	1
and	5
by	1
clusters	1
companies	1
computers.	1
data	2
datasets	1
distributed	1
each	1
ecosystem	1
emits	1
for	3
framework	1
function	2
hello	1
in	1
includes	1
input	1
is	5
key	1
key.	1
large	2
like	1
many	1
model	1
of	2
open	1
pairs.	1
process	1
processes	1
processing	1
processing.	1
programming	1
programming.	1
sets.	1
source	1
storage	1
the	2
to	1
tools	1
used	1
uses	1
value	1
values	1
world	1
worldwide.	1
written	1


In [17]:
# Visualizar los resultados de forma ordenada
import subprocess

result = subprocess.run(
    ["hdfs", "dfs", "-cat", "/user/root/wordcount/output/part-r-00000"],
    capture_output=True, text=True
)

# Parsear y ordenar por frecuencia
word_counts = []
for line in result.stdout.strip().split("\n"):
    if line.strip():
        parts = line.split("\t")
        if len(parts) == 2:
            word_counts.append((parts[0], int(parts[1])))

word_counts.sort(key=lambda x: x[1], reverse=True)

print(f"{'Palabra':<25} {'Frecuencia':>10}")
print("-" * 37)
for word, count in word_counts:
    print(f"{word:<25} {count:>10}")

Palabra                   Frecuencia
-------------------------------------
Hadoop                             5
and                                5
is                                 5
MapReduce                          3
The                                3
for                                3
data                               2
function                           2
large                              2
of                                 2
the                                2
HBase,                             1
Hive,                              1
Java                               1
Map                                1
Pig,                               1
Reduce                             1
Spark.                             1
WordCount                          1
a                                  1
across                             1
aggregates                         1
an                                 1
by                                 1
clusters                           1


## 9. (Opcional) WordCount con Hadoop Streaming y Python

También podemos ejecutar WordCount usando **Hadoop Streaming** con scripts en Python, sin necesidad de Java.

In [18]:
# Mapper en Python
mapper_code = '''#!/usr/bin/env python3
import sys

for line in sys.stdin:
    line = line.strip().lower()
    words = line.split()
    for word in words:
        # Limpiar puntuación
        word = ''.join(c for c in word if c.isalpha())
        if word:
            print(f"{word}\t1")
'''

with open("/tmp/mapper.py", "w") as f:
    f.write(mapper_code)
!chmod +x /tmp/mapper.py

print("✅ mapper.py creado")
print(mapper_code)

✅ mapper.py creado
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    line = line.strip().lower()
    words = line.split()
    for word in words:
        # Limpiar puntuación
        word = ''.join(c for c in word if c.isalpha())
        if word:
            print(f"{word}	1")



In [19]:
# Reducer en Python
reducer_code = '''#!/usr/bin/env python3
import sys
from itertools import groupby
from operator import itemgetter

def read_mapper_output(file):
    for line in file:
        yield line.strip().split("\t", 1)

data = read_mapper_output(sys.stdin)

for current_word, group in groupby(data, itemgetter(0)):
    total_count = sum(int(count) for _, count in group)
    print(f"{current_word}\t{total_count}")
'''

with open("/tmp/reducer.py", "w") as f:
    f.write(reducer_code)
!chmod +x /tmp/reducer.py

print("✅ reducer.py creado")
print(reducer_code)

✅ reducer.py creado
#!/usr/bin/env python3
import sys
from itertools import groupby
from operator import itemgetter

def read_mapper_output(file):
    for line in file:
        yield line.strip().split("	", 1)

data = read_mapper_output(sys.stdin)

for current_word, group in groupby(data, itemgetter(0)):
    total_count = sum(int(count) for _, count in group)
    print(f"{current_word}	{total_count}")



In [20]:
# Primero probamos localmente sin Hadoop
print("🧪 Prueba local del mapper y reducer:")
!cat /tmp/input1.txt | python3 /tmp/mapper.py | sort | python3 /tmp/reducer.py

🧪 Prueba local del mapper y reducer:
across	1
an	1
and	3
by	1
clusters	1
companies	1
computers	1
datasets	1
distributed	1
ecosystem	1
for	1
framework	1
hadoop	4
hbase	1
hive	1
in	1
includes	1
is	3
java	1
large	1
like	1
many	1
mapreduce	1
of	1
open	1
pig	1
process	1
processing	1
source	1
spark	1
storage	1
the	1
to	1
tools	1
used	1
uses	1
worldwide	1
written	1


In [ ]:
# Ejecutar con Hadoop Streaming
STREAMING_JAR = "/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-*.jar"

# Eliminar output previo si existe
!hdfs dfs -rm -r -f /user/root/wordcount/output_streaming

!hadoop jar {STREAMING_JAR} \
    -files /tmp/mapper.py,/tmp/reducer.py \
    -mapper mapper.py \
    -reducer reducer.py \
    -input /user/root/wordcount/input \
    -output /user/root/wordcount/output_streaming

packageJobJar: [/tmp/hadoop-unjar14787997771470732978/] [] /tmp/streamjob15449417888367939820.jar tmpDir=null
2026-03-02 06:48:56,689 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-03-02 06:48:57,024 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-03-02 06:48:57,330 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1772434026935_0002
2026-03-02 06:48:57,856 INFO mapred.FileInputFormat: Total input files to process : 2
2026-03-02 06:48:58,344 INFO mapreduce.JobSubmitter: number of splits:2
2026-03-02 06:48:59,090 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1772434026935_0002
2026-03-02 06:48:59,090 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-03-02 06:48:59,367 INFO conf.Configuration: resource-types.xml not found
2026-03-02 06:48:59,368 INFO resource.ResourceUtils: Unable to find 'resource

In [ ]:
# Ver resultados del Streaming
print("📊 Resultado WordCount con Hadoop Streaming (Python):")
!hdfs dfs -cat /user/root/wordcount/output_streaming/part-00000 | sort -k2 -rn | head -20

## 10. Detener los Servicios de Hadoop

In [ ]:
# Detener YARN y HDFS
!stop-yarn.sh
!stop-dfs.sh
print("✅ Hadoop detenido correctamente")